# Parte I — Fundamentos · o código dos capítulos 05 e 06

**Este caderno não contém o algoritmo.** Ele busca o código publicado do handbook e chama as
mesmas funções que o `pytest` do repositório verifica. Se o código mudar, o caderno muda junto:
não existe segunda cópia para envelhecer — é a regra da
[ADR 0016](https://github.com/GHDaru/operationalresearchaibook/blob/main/adr/0016-cadernos-colab-sem-deriva.md).

Os capítulos 01 a 04 não aparecem aqui, e isso é declarado em vez de disfarçado: eles não
produzem número próprio, então não há o que rodar.

Rode as células em ordem. No Colab, `Ctrl+F9` roda tudo.

> **Ele roda igual na sua máquina.** O Colab é conveniência, não dependência: o experimento é o
> script em `po-zero/`, que roda em qualquer CPU sem licença paga.


In [ ]:
# 1. Traz o código publicado. Sem magias do IPython (`!git`, `%cd`): Python puro
#    roda igual no Colab e no seu terminal — e é o que o teste consegue executar.
import subprocess, sys
from pathlib import Path

URL = "https://github.com/GHDaru/operationalresearchaibook"
RAIZ = Path("operationalresearchaibook")

if not RAIZ.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(RAIZ)], check=True)

ETAPA = RAIZ / "po-zero" / "parte-I-fundamentos"
sys.path.insert(0, str(ETAPA.resolve()))
print("código em:", ETAPA.resolve())

## Capítulo 05 — o pior caso, construído

O cubo de Klee–Minty é um cubo de $n$ dimensões levemente entortado, com $2^n$ vértices. Com a
regra de pivoteamento clássica, o Simplex passa por **todos** eles.

Isto não é citação: é construção. A célula abaixo monta o cubo e conta os pivôs, em aritmética
exata.


In [ ]:
# 2. O pior caso do Simplex, medido em vez de citado.
from complexidade import MAGNITUDES, perfil_aleatorio, pior_caso

for n in range(2, 8):
    p = pior_caso(n)
    print(f"n={n}  vertices={p['vertices']:>4}  pivos={p['pivos']:>4}  (2^n-1 = {p['esperado']})")
    assert p["pivos"] == 2 ** n - 1

## O palpite — erre antes de ver o certo

Circula com esse resultado uma frase tranquilizadora: *"tudo bem, o pior caso é frágil; qualquer
perturbaçãozinha desmancha"*.

**Antes de rodar a próxima célula, decida:** o cubo com $n = 6$ custa 63 pivôs. Se eu perturbar a
matriz de restrições em **1%**, quantos pivôs sobram?

Escreva a sua resposta em algum lugar. A célula seguinte roda o palpite mais comum e depois o que de fato acontece — nesta ordem, de
propósito. E mede sobre **200 sementes**, porque uma perturbação é um sorteio e um sorteio não é
uma medição: esta tabela já foi publicada errada duas vezes por esquecer isso.

In [ ]:
# 3. Primeiro o palpite. Depois a realidade.
from complexidade import AMOSTRAS_PERTURBACAO, perfil_de_perturbacao, varredura_em_n
from fractions import Fraction as F

N = 6
puro = 2 ** N - 1
print(f"o palpite comum diz: perturbou 1%, o caminho encurta muito -> algo bem abaixo de {puro}")
print()
print(f"{AMOSTRAS_PERTURBACAO} sementes por magnitude -- porque um sorteio nao e uma medicao:")
print()

for mag in MAGNITUDES:
    p = perfil_de_perturbacao(N, mag)
    marca = "  <-- nada mudou, em NENHUMA semente" if p["intactas"] == p["amostras"] else ""
    print(f"perturbacao de {str(mag):>6}: mediana {p['mediana']:>3} pivos  "
          f"intactos {p['intactas']:>3}/{p['amostras']}{marca}")

print()
print("0,1% e 1% nao mudam NADA, nas 200. A 10% o caminho as vezes quebra;")
print("so a 25% e 50% ele quebra com regularidade.")
print()
print("E o eixo que a tabela de n=6 nao olha -- o teorema de 2004 vive AQUI:")
for mag in (F(1, 100), F(1, 10)):
    linhas = varredura_em_n(mag)
    print(f"  {str(mag):>6}: " + "  ".join(f"n={l['n']}: {l['intactas']}/{l['amostras']}" for l in linhas))

> **O que isto não autoriza concluir.** A medição **não** refuta a análise suavizada de Spielman
> e Teng (2004). Aquele teorema é assintótico, vale **em esperança**, supõe perturbação
> **gaussiana** e uma regra de pivoteamento específica — nenhuma das três condições vale acima.
>
> O que ela refuta é a **frase de corredor**, que trata um teorema delicado como se dissesse
> "mexeu um pouco, melhorou". Não diz.


## E a sua instância? — mexa aqui

Do outro lado da distância está o número que decide projeto: quantos pivôs custa uma instância
**sem malícia**, do mesmo tamanho do cubo.

Troque `MEU_N` abaixo e rode de novo. Tente 5, 10, 15 e 20, e olhe as duas últimas colunas lado
a lado — é a distância entre a teoria e a mesa de trabalho.


In [ ]:
# 4. Sua vez. Mude o numero e rode.
MEU_N = 10      # <<< mexa aqui

p = perfil_aleatorio(MEU_N)
print(f"n=m={p['n']}  em {p['amostras']} instancias aleatorias:")
print(f"  mediana {p['mediana']} pivos · a PIOR das {p['amostras']} custou {p['maximo']}")
print(f"  o cubo de Klee-Minty do mesmo tamanho custa: {p['pior_caso_construido']}")
print()
print("Repare no rotulo: 'o cubo do mesmo tamanho', e nao 'pior caso teorico'.")
print("2^n-1 e um pior caso CONSTRUIDO para uma regra de pivoteamento -- nao e")
print("o numero de vertices que um poliedro daquele tamanho pode ter.")
print()
print("E cuidado com o que isto sustenta: instancia aleatoria NAO e instancia real.")
print("O que a tabela nega e a INFERENCIA -- da classe nao se deduz o custo da sua.")

## Capítulo 06 — a ferramenta escolhe o seu plano

Um modelo minúsculo, com face ótima inteira: maximizar $A + B$ sujeito a $A + B \le 10$,
$A \le 6$, $B \le 8$. Todo ponto do segmento entre $(2,8)$ e $(6,4)$ vale 10.

**Antes de rodar:** o Simplex exato e o solver vão devolver o mesmo plano?


In [ ]:
# 5. Mesmo modelo, mesmo valor, planos diferentes.
from ferramentas import multiplos_otimos, racao, vereditos

m = multiplos_otimos()
print(f"Simplex exato : A={m['exato']['ponto'][0]}, B={m['exato']['ponto'][1]}  · valor {m['exato']['valor']}")
for s in m["solvers"]:
    print(f"{s['solver']:<14}: A={s['ponto'][0]}, B={s['ponto'][1]}  · valor {s['valor']}")

print()
print("O relatorio diz 10 nos tres. O plano que alguem vai EXECUTAR, nao.")
assert m["exato"]["ponto"] == ["6", "4"]
assert all([round(v) for v in s["ponto"]] == [2, 8] for s in m["solvers"])

In [ ]:
# 6. E nenhum solver devolve a fracao.
r = racao()
print(f"exato : {r['exato']['valor']}  (= {r['valor_exato_float']})")
for s in r["solvers"]:
    print(f"{s['solver']:<6}: {s['valor']!r}  · erro {s['erro_absoluto']:.2e}")

print()
print("Os dois estao certos e discordam entre si. Nunca compare saida de solver com ==.")

v = vereditos()
print()
print("Ja os VEREDITOS concordam -- e isso tambem e resultado:")
for nome, d in v.items():
    print(f"  {nome:<10}: exato={d['exato']:<11} solvers={d['solvers']}")
print("Infeasible nao se conserta trocando de solver. O problema e o modelo.")

## O que o livro não mostra

A célula seguinte imprime **o código-fonte** da função que perturba o cubo. O capítulo explica o
resultado; aqui você lê a implementação — inclusive o comentário que registra, na própria função,
o que a medição **não** autoriza concluir.


In [ ]:
# 7. O algoritmo, lido em vez de descrito.
import inspect, complexidade
print(inspect.getsource(complexidade.perturba))